# Reproduce — Retained Spectral (credibility audit)

One click reproduces the whole benchmark on a fresh machine and **certifies its
credibility from the numbers measured here** — nothing is copied from the repo's
committed results.

Two axes are kept strictly separate (the project's tier-honesty rule):

1. **Credibility** — is the reproduction sound? Correctness at the *declared* tolerance on
   an 8-case adversarial suite (wells translated far off-origin, narrow/broad wells, an
   8-mode Morse well, a factorized double well, quartic scale covariance over 5 decades),
   a 20-point mode×grid scaling sweep cross-checked against SciPy, a recorded source commit,
   and a frozen single-thread environment. **This is what the audit ACCEPTs.**
2. **Speed / fairness** — does the native **Retained Multilevel Sturm** solver beat every
   standard eigensolver? Reported as `baseline_benchmark_verdict`, and it may honestly be
   **HOLD** when a competitor (e.g. ARPACK) fails to converge on a well — that never lowers
   the reproduction's credibility, and a green audit never implies universal dominance.

Tier: `finite_diagnostic` — a discrete rational-readout agreement-and-cost claim, not a
continuum-limit proof or an empirical-physics claim.


## 1 · Frozen environment

Pinned numeric stack (bit-reproducible) + single-thread BLAS, set **before** numpy is
imported. The package is installed from `main`; the exact resolved source commit is printed
and also recorded inside every audit JSON (`environment.github_sha` / `source_commit`).


In [ ]:
# single-thread BLAS/LAPACK — a timing comparison must measure the algorithm, not the
# host thread scheduler. Set before numpy is imported anywhere.
import os
for _v in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS',
           'VECLIB_MAXIMUM_THREADS','NUMEXPR_NUM_THREADS'):
    os.environ[_v] = '1'

# pinned numeric stack (see requirements-spectral-lock.txt in the repo), then the package
!pip -q install numpy==1.26.4 scipy==1.13.1 numba==0.60.0 llvmlite==0.43.0 matplotlib==3.9.2
!pip -q install "information-discrete-math[spectral-bench] @ git+https://github.com/morrocwi/information-discrete-math"

import importlib.metadata as _md
print('information-discrete-math', _md.version('information-discrete-math'))
!pip show -f information-discrete-math 2>/dev/null | grep -i '^Version' || true

## 2 · Credibility audit (measured live)

Runs the consolidated `credibility_audit`: the 7-case baseline competition + 8 adversarial
cases + a 20-point scaling sweep + cold-start, all against the same **independent,
reference-blind** SciPy pipeline (its well search is unbounded — it is not handed the
native window). ACCEPT here means the reproduction is correct and reproducible.


In [ ]:
import json
from pathlib import Path
from retained_spectral.competition.credibility_audit import run_credibility_audit

audit = run_credibility_audit(include_jax=False, baseline_repeats=9,
                              executor_repeats=5, scaling_repeats=7)
Path('credibility-audit.json').write_text(json.dumps(audit, indent=2))

print('credibility_gates:', json.dumps(audit['credibility_gates'], indent=2))
print('credibility verdict           :', audit['verdict'])
print('baseline benchmark verdict    :', audit['baseline_benchmark_verdict'], '(reported, not a credibility gate)')
print('adversarial cases all correct :', audit['adversarial']['all_ok'], f"({len(audit['adversarial']['cases'])} cases)")
print('scaling cross-checks all ok   :', audit['scaling']['all_cross_checks'], f"({len(audit['scaling']['cases'])} points)")
print('source commit recorded        :', audit['baseline']['source_commit'])

## 3 · Headline competition + chart (measured live)

The 7 declared spectra: native RMS against every standard eigensolver on one identical
operator, timed with seeded-randomized ordering, `verdict_gates` shown in full. The chart
is redrawn from the numbers just measured.


In [ ]:
import json
from pathlib import Path
from retained_spectral.competition.run import run_competition

result = run_competition(repeats=9, audit_repeats=5, include_jax=False)  # live on THIS machine
Path('results.json').write_text(json.dumps(result, indent=2))
print(json.dumps(result['end_to_end']['summary'], indent=2))
print('verdict_gates:', json.dumps(result['verdict_gates'], indent=2))
print('overall verdict:', result['verdict'], '| seed:', result['end_to_end']['seed'])

In [ ]:
from pathlib import Path
from retained_spectral.competition.chart import render_hero, render_detail

render_hero(result, Path('hero.png'))
render_detail(result, Path('detail.png'))
from IPython.display import Image, display
display(Image('hero.png'))
display(Image('detail.png'))

## 4 · Solve your own problem from raw input


In [ ]:
import retained_spectral as rs
problem = rs.make_problem(name='my', family='harmonic',
                          parameters={'omega': 2.0, 'center': 0.0}, modes=4)
r = rs.solve(problem)
print(r.status, r.values)   # ACCEPT (1.0, 3.0, 5.0, 7.0)